# 3. Model Explainability (SHAP)



In [ ]:
import pandas as pd
import pickle
import shap
import matplotlib.pyplot as plt

# Load model and data
with open('../models/xgboost_best.pkl', 'rb') as f:
    xgb_model = pickle.load(f)

df = pd.read_csv('../data/customer_churn_data.csv')
df.drop('customerID', axis=1, inplace=True)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Feature Engineering
df['AvgMonthlySpend'] = df.apply(lambda row: row['TotalCharges']/row['tenure'] if row['tenure']>0 else 0, axis=1)
df['Tenure_Bucket'] = pd.cut(df['tenure'], bins=[-1, 12, 48, 100], labels=['New', 'Medium', 'Loyal'])

services = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['Service_Count'] = sum(df[svc].apply(lambda x: 1 if x in ['Yes'] else 0) for svc in services)
df['Is_LongTerm_Contract'] = df['Contract'].apply(lambda x: 1 if x in ['One year', 'Two year'] else 0)
high_val_thresh = df['MonthlyCharges'].quantile(0.8)
df['High_Value_Customer'] = df['MonthlyCharges'].apply(lambda x: 1 if x >= high_val_thresh else 0)

X = df.drop('Churn', axis=1)

with open('../models/label_encoders.pkl', 'rb') as f:
    le_dict = pickle.load(f)
with open('../models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('../models/columns.pkl', 'rb') as f:
    cols = pickle.load(f)

for col, le in le_dict.items():
    X[col] = X[col].astype(str)
    # Handle unseen
    X[col] = X[col].apply(lambda x: x if x in le.classes_ else le.classes_[0])
    X[col] = le.transform(X[col])

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'Service_Count']
X[num_cols] = scaler.transform(X[num_cols])
X = X[cols] # Ensure column order



## Global Interpretability



In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X.sample(1000, random_state=42))

shap.summary_plot(shap_values, X.sample(1000, random_state=42))



## Local Interpretability (Single Customer)



In [ ]:
shap.force_plot(explainer.expected_value, shap_values[0,:], X.iloc[0,:], matplotlib=True)
plt.savefig('local_explanation.png')

